# Pourquoi le taux de chômage reste-t-il élevé au Maroc ?
Notebook d'analyse : données Banque mondiale (API, séries longues) + tableaux du HCP (saisis à partir des notes d'information officielles).
Exécutez les cellules dans l'ordre (Exécution > Tout exécuter).

In [ ]:
# 1) Imports et réglages
import os, warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.makedirs("figures", exist_ok=True)
pd.options.display.float_format = "{:,.1f}".format
plt.rcParams.update({
    "figure.figsize": (10, 5.5), "axes.grid": True, "grid.alpha": .3,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titleweight": "bold",
})

def sauver(nom):
    plt.tight_layout()
    plt.savefig(f"figures/{nom}.png", dpi=150)
    plt.show()

In [ ]:
# 2) Téléchargement des données de la Banque mondiale (estimations modélisées OIT)
INDICATEURS = {
    "SL.UNE.TOTL.ZS":    "Chômage total",
    "SL.UNE.TOTL.FE.ZS": "Chômage femmes",
    "SL.UNE.TOTL.MA.ZS": "Chômage hommes",
    "SL.UEM.1524.ZS":    "Chômage jeunes 15-24",
    "SL.UEM.BASC.ZS":    "Chômage éducation de base",
    "SL.UEM.INTM.ZS":    "Chômage éducation intermédiaire",
    "SL.UEM.ADVN.ZS":    "Chômage éducation avancée",
    "NY.GDP.MKTP.KD.ZG": "Croissance PIB",
    "NE.GDI.FTOT.ZS":    "Investissement (% PIB)",
    "SL.TLF.CACT.ZS":    "Taux d'activité total",
    "SL.TLF.CACT.FE.ZS": "Taux d'activité femmes",
    "SL.AGR.EMPL.ZS":    "Emploi agriculture",
    "SL.IND.EMPL.ZS":    "Emploi industrie",
    "SL.SRV.EMPL.ZS":    "Emploi services",
    "SL.EMP.VULN.ZS":    "Emploi vulnérable",
    "SL.UEM.NEET.ZS":    "Jeunes NEET",
    "SP.POP.1564.TO.ZS": "Population 15-64 ans (% total)",
}

def wb(code, pays="MAR", debut=1991, fin=2025):
    url = f"https://api.worldbank.org/v2/country/{pays}/indicator/{code}"
    r = requests.get(url, params={"format": "json", "per_page": 1000, "date": f"{debut}:{fin}"}, timeout=30)
    r.raise_for_status()
    js = r.json()
    if len(js) < 2 or js[1] is None:
        return pd.Series(dtype=float)
    return pd.Series({int(d["date"]): d["value"] for d in js[1]}, dtype=float).sort_index()

series = {}
for code, nom in INDICATEURS.items():
    try:
        s = wb(code)
        if s.dropna().empty:
            print(f"✗ {nom:35s} (aucune donnée)")
        else:
            series[nom] = s
            print(f"✓ {nom:35s} {int(s.dropna().index.min())}-{int(s.dropna().index.max())}")
    except Exception as e:
        print(f"✗ {nom:35s} erreur : {e}")

df = pd.DataFrame(series).sort_index()
df.index.name = "Année"
df.to_csv("donnees_banque_mondiale_maroc.csv")
df.tail(10)

In [ ]:
# 3) Graphique 1 : le chômage augmente-t-il vraiment « sans cesse » ?
s = df["Chômage total"].dropna()
fig, ax = plt.subplots()
ax.plot(s.index, s.values, marker="o", lw=2, color="#c0392b", label="Taux de chômage")
pente = np.polyfit(s.index, s.values, 1)
ax.plot(s.index, np.polyval(pente, s.index), "--", color="gray", label=f"Tendance linéaire ({pente[0]:+.2f} pt/an)")
for x, txt in {2008: "Crise financière", 2020: "Covid-19"}.items():
    if x in s.index:
        ax.axvline(x, ls=":", color="k", alpha=.5)
        ax.text(x + .2, s.max(), txt, rotation=90, va="top", fontsize=9)
ax.set_title("Taux de chômage au Maroc (% de la population active)")
ax.set_ylabel("%"); ax.legend()
sauver("01_chomage_total")

# Tableau : moyenne et tendance par sous-période
periodes = [(1991, 1999), (2000, 2009), (2010, 2019), (2020, 2025)]
lignes = []
for a, b in periodes:
    sp = s[(s.index >= a) & (s.index <= b)]
    if len(sp) >= 3:
        lignes.append({"Période": f"{a}-{b}", "Moyenne (%)": sp.mean(), "Min (%)": sp.min(),
                       "Max (%)": sp.max(), "Tendance (pt/an)": np.polyfit(sp.index, sp.values, 1)[0]})
tab_periodes = pd.DataFrame(lignes).set_index("Période")
tab_periodes

In [ ]:
# 4) Graphique 2 : qui est le plus touché ? (sexe, jeunes)
cols = [c for c in ["Chômage total", "Chômage hommes", "Chômage femmes", "Chômage jeunes 15-24"] if c in df]
couleurs = {"Chômage total": "#2c3e50", "Chômage hommes": "#2980b9", "Chômage femmes": "#e67e22", "Chômage jeunes 15-24": "#c0392b"}
fig, ax = plt.subplots()
for c in cols:
    ax.plot(df.index, df[c], lw=2, label=c.replace("Chômage ", ""), color=couleurs[c])
ax.set_title("Chômage : total, hommes, femmes et jeunes (15-24 ans)")
ax.set_ylabel("%"); ax.legend()
sauver("02_sexe_jeunes")

dernier = df[cols].dropna(how="all").iloc[-1]
print(f"Dernière année disponible : {df[cols].dropna(how='all').index[-1]}")
dernier.to_frame("Taux (%)")

In [ ]:
# 5) Graphique 3 : croissance économique vs chômage (loi d'Okun)
if "Croissance PIB" in df:
    d = df[["Croissance PIB", "Chômage total"]].dropna().copy()
    d["Variation du chômage (pt)"] = d["Chômage total"].diff()
    d = d.dropna()

    fig, ax1 = plt.subplots()
    ax1.bar(d.index, d["Croissance PIB"], color="#95a5a6", alpha=.7, label="Croissance du PIB (%)")
    ax1.set_ylabel("Croissance du PIB (%)")
    ax2 = ax1.twinx()
    ax2.plot(d.index, d["Chômage total"], color="#c0392b", lw=2, marker="o", label="Chômage (%)")
    ax2.set_ylabel("Chômage (%)"); ax2.grid(False)
    ax1.set_title("Croissance du PIB et chômage")
    fig.legend(loc="upper right", bbox_to_anchor=(.9, .88))
    sauver("03_pib_chomage")

    fig, ax = plt.subplots()
    ax.scatter(d["Croissance PIB"], d["Variation du chômage (pt)"], color="#2980b9")
    k = np.polyfit(d["Croissance PIB"], d["Variation du chômage (pt)"], 1)
    xs = np.linspace(d["Croissance PIB"].min(), d["Croissance PIB"].max(), 50)
    ax.plot(xs, np.polyval(k, xs), "--", color="#c0392b")
    r = d["Croissance PIB"].corr(d["Variation du chômage (pt)"])
    ax.set_title(f"Loi d'Okun : croissance vs variation du chômage (r = {r:.2f})")
    ax.set_xlabel("Croissance du PIB (%)"); ax.set_ylabel("Variation annuelle du chômage (points)")
    ax.axhline(0, color="k", lw=.8)
    sauver("04_okun")
    print(f"Corrélation = {r:.2f} → une corrélation faible signifie que la croissance crée peu d'emplois (croissance peu riche en emplois).")

In [ ]:
# 6) Graphique 4 : structure de l'économie et de l'emploi
sect = [c for c in ["Emploi agriculture", "Emploi industrie", "Emploi services"] if c in df]
if sect:
    dd = df[sect].dropna()
    fig, ax = plt.subplots()
    ax.stackplot(dd.index, [dd[c] for c in sect], labels=[c.replace("Emploi ", "") for c in sect],
                 colors=["#27ae60", "#7f8c8d", "#2980b9"], alpha=.85)
    ax.set_title("Répartition de l'emploi par secteur (%)"); ax.set_ylim(0, 100); ax.legend(loc="upper right")
    sauver("05_secteurs")

part = [c for c in ["Taux d'activité total", "Taux d'activité femmes", "Emploi vulnérable"] if c in df]
if part:
    fig, ax = plt.subplots()
    for c in part:
        ax.plot(df.index, df[c], lw=2, label=c)
    ax.set_title("Participation au marché du travail et qualité de l'emploi (%)"); ax.legend()
    sauver("06_activite_vulnerable")

In [ ]:
# 7) Graphique 5 : le paradoxe du diplômé (chômage par niveau d'éducation)
edu = [c for c in ["Chômage éducation de base", "Chômage éducation intermédiaire", "Chômage éducation avancée"] if c in df]
if edu:
    fig, ax = plt.subplots()
    for c in edu:
        ax.plot(df.index, df[c], lw=2, label=c.replace("Chômage éducation ", "Éducation "))
    ax.set_title("Chômage selon le niveau d'éducation (% de la population active du groupe)")
    ax.set_ylabel("%"); ax.legend()
    sauver("07_education")

In [ ]:
# 8) Matrice de corrélations : quels facteurs vont avec le chômage ?
num = df.dropna(axis=1, thresh=15)
corr = num.corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns, fontsize=8)
ax.grid(False); fig.colorbar(im, shrink=.8)
ax.set_title("Corrélations entre indicateurs (attention : corrélation ≠ causalité)")
sauver("08_correlations")
corr["Chômage total"].drop("Chômage total").sort_values().to_frame("Corrélation avec le chômage total")

## Partie 2 : données officielles du HCP (Enquête nationale sur l'emploi)
Les chiffres ci-dessous proviennent des notes d'information du HCP sur la situation du marché du travail (année 2025 vs 2024).
Vérifiez-les et complétez-les sur https://www.hcp.ma (rubrique « Emploi »).

In [ ]:
# 9) Tableau HCP 2024 vs 2025 par catégorie
hcp = pd.DataFrame({
    "Catégorie": ["Ensemble", "Urbain", "Rural", "Hommes", "Femmes",
                  "15-24 ans", "25-34 ans", "35-44 ans", "45 ans et +",
                  "Diplômés", "Sans diplôme",
                  "Techniciens / cadres moyens", "Qualification professionnelle"],
    "Groupe": ["Ensemble", "Milieu", "Milieu", "Sexe", "Sexe",
               "Âge", "Âge", "Âge", "Âge", "Diplôme", "Diplôme", "Diplôme", "Diplôme"],
    "2024": [13.3, 16.9, 6.8, 11.6, 19.4, 36.7, 21.0, 7.6, 4.0, 19.6, 5.2, 26.3, 23.9],
    "2025": [13.0, 16.4, 6.6, 10.8, 20.5, 37.2, 20.9, 7.2, 3.6, 19.1, 4.7, 24.0, 22.0],
})
hcp["Écart (pts)"] = hcp["2025"] - hcp["2024"]
hcp.set_index("Catégorie")

In [ ]:
# 10) Graphique 6 : taux de chômage par catégorie (2025) — qui est touché ?
h = hcp[hcp["Groupe"] != "Ensemble"].sort_values("2025")
fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(h))
ax.barh(y - .2, h["2024"], height=.4, color="#bdc3c7", label="2024")
ax.barh(y + .2, h["2025"], height=.4, color="#c0392b", label="2025")
ax.set_yticks(y); ax.set_yticklabels(h["Catégorie"])
ax.axvline(13.0, color="k", ls="--", lw=1); ax.text(13.2, 0, "Moyenne nationale 2025 : 13,0 %", fontsize=9)
ax.set_xlabel("Taux de chômage (%)"); ax.set_title("HCP : chômage par catégorie, 2024 vs 2025"); ax.legend(loc="lower right")
sauver("09_hcp_categories")

In [ ]:
# 11) Graphique 7 : nature du chômage (primo-demandeurs, longue durée, durée moyenne)
nat = pd.DataFrame({
    "Indicateur": ["Chômeurs n'ayant jamais travaillé (%)", "Chômeurs depuis 1 an ou plus (%)", "Durée moyenne du chômage (mois)"],
    "2024": [49.3, 62.4, 31], "2025": [52.9, 64.8, 33],
}).set_index("Indicateur")
display(nat)
ax = nat.T.plot(kind="bar", figsize=(10, 5.5), color=["#2980b9", "#c0392b", "#7f8c8d"], rot=0)
ax.set_title("Le chômage devient plus structurel : plus long et plus concentré sur les primo-demandeurs")
sauver("10_nature_chomage")

In [ ]:
# 12) Graphique 8 : disparités régionales (T1 2025, note HCP)
reg = pd.Series({"Oriental": 25.2, "Régions du Sud": 23.8, "Tanger-Tétouan-Al Hoceima": 11.2,
                 "Marrakech-Safi": 8.9, "Drâa-Tafilalet": 8.0}).sort_values()
fig, ax = plt.subplots()
ax.barh(reg.index, reg.values, color="#e67e22")
for i, v in enumerate(reg.values):
    ax.text(v + .3, i, f"{v:.1f}%", va="center")
ax.set_title("Taux de chômage par région (T1 2025, régions citées par le HCP)"); ax.set_xlabel("%")
sauver("11_regions")
# Astuce : ajoutez les 7 autres régions depuis le tableau du HCP pour un graphique complet.

In [ ]:
# 13) Graphique 9 : au-delà du chômage « strict » (nouvelle enquête EMO, T1 2026)
# Attention : la méthodologie a changé en 2026, ces chiffres ne sont PAS comparables à la série précédente.
emo = pd.DataFrame({
    "Chômage strict":                         [10.8, 13.5, 6.1, 9.4, 16.1],
    "+ sous-emploi (durée du travail)":       [16.6, 18.3, 13.6, 15.7, 19.8],
    "+ main-d'œuvre potentielle":             [17.1, 20.4, 11.2, 13.9, 27.9],
}, index=["National", "Urbain", "Rural", "Hommes", "Femmes"])
display(emo)
ax = emo.plot(kind="bar", figsize=(10, 5.5), color=["#c0392b", "#e67e22", "#7f8c8d"], rot=0)
ax.set_ylabel("%"); ax.set_title("HCP T1 2026 : trois mesures de la sous-utilisation de la main-d'œuvre")
sauver("12_emo_2026")

In [ ]:
# 14) Export : tableaux + graphiques
with pd.ExcelWriter("tableaux_chomage_maroc.xlsx") as xl:
    df.to_excel(xl, sheet_name="Banque mondiale")
    tab_periodes.to_excel(xl, sheet_name="Periodes")
    hcp.to_excel(xl, sheet_name="HCP 2024-2025", index=False)
    nat.to_excel(xl, sheet_name="Nature chomage")
    emo.to_excel(xl, sheet_name="EMO 2026")
os.system("zip -qr figures.zip figures")
try:
    from google.colab import files
    files.download("tableaux_chomage_maroc.xlsx")
    files.download("figures.zip")
except ImportError:
    print("Fichiers créés : tableaux_chomage_maroc.xlsx et figures.zip")